In [2]:
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()

In [4]:
import sys
sys.path.insert(0, "..")

In [6]:
from utils.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [7]:
from utils.rag_helper import RAGBase

instructions = '''

You're a course teaching assistant. 
Answer the Question based on the context from the FAQ database.
Use only the facts from the context when answering the question
'''.strip()

assistant = RAGBase(
    index = index,
    llm_client = client,
    instructions = instructions
)

In [8]:

messages = [{"role": "user", "content": 'I just discovered the course. can I join it ?'}]
model = "claude-opus-5-5"
response = client.messages.create(
    model=model,
    max_tokens=1024,
    system=instructions,
    messages=messages,
        )

response = next(b.text for b in response.content if b.type == "text")


In [9]:
response

"I'm sorry, but I don't have any FAQ context to answer your question. No course information or FAQ entries were included with your message, and I can only answer using facts from that material.\n\nIf you share the relevant FAQ section, especially anything about enrollment, deadlines, or late registration, I can tell you whether you can still join and how."

In [16]:
def search(query):
    boost_dict = {'question':3.0, 'section': 0.5}
    filter_dict = {'course':'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [17]:
search_tool = {
    "type": "custom",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [20]:
import json

messages = [{"role": "user", "content": "I just discovered the course. can I join it ?"}]

while True:
    response = client.messages.create(
        model="claude-opus-5-5",
        max_tokens=16000,
        system=instructions,
        messages=messages,
        tools=[search_tool],
    )
    messages.append({"role": "assistant", "content": response.content})

    if response.stop_reason != "tool_use":
        break

    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            results = search(**block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(results, default=str),
            })
    messages.append({"role": "user", "content": tool_results})

answer = next(b.text for b in response.content if b.type == "text")
print(answer)


Yes, you can still join. If you want a certificate, you'll need to submit your project while submissions are still open.

**Getting started:**
- Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/) and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).
- You can start whenever you like. The videos and GitHub materials are already available, and deadlines are listed on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

**Weekly routine:**
1. Watch the lesson videos.
2. Work through the lesson notebooks and code.
3. Read the homework instructions on GitHub.
4. Submit your answers on the course platform before the deadline.

**If you've missed homework:**
- Homework is optional. You can still get the certificate as long as you pass the Capstone project.
- Doing the homework is still a good idea

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

In [ ]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break 